# IAN 603 · Module 7
# Subqueries in PostgreSQL

Use one query inside another for benchmarks, membership, existence, row-dependent comparisons, and reusable intermediate results.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module7.html


## Learning goals

Work through the examples in order. Focus on what each SQL pattern does, what one output row represents, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database
db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script
print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**. You may save the connection settings on your own computer for the next run.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 7

Choose a subquery pattern based on the **shape** the outer query needs.

| Need | Common pattern |
|---|---|
| one value | scalar subquery |
| a set of values | `IN` |
| whether a matching row exists | `EXISTS` |
| compare with any/all values | `ANY` / `ALL` |
| value depends on current outer row | correlated subquery |
| reusable table-like result | derived table or CTE |

### Important NULL lesson

`NOT IN` can behave unexpectedly when the inner result contains `NULL`.

For anti-matching logic, `NOT EXISTS` is often easier to reason about safely.


# Teaching database

Run the next cell to create or reset the Module 7 teaching data. Re-run it whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 7\n-- Subqueries in PostgreSQL\n-- ============================================================\n-- Companion file for s_module7.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- scalar and multi-row subqueries, IN, EXISTS, NOT EXISTS,\n-- correlated subqueries, derived tables, nested logic, and CTEs.\n--\n-- Choose a pattern based on the shape of the value or row set\n-- needed by the outer query.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS review_queue CASCADE;\nDROP TABLE IF EXISTS prescriptions CASCADE;\nDROP TABLE IF EXISTS lab_results CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS departments CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n    patient_id integer PRIMARY KEY,\n    name varchar(100) NOT NULL,\n    age integer NOT NULL CHECK (age BETWEEN 0 AND 120),\n    gender varchar(20),\n    home_city varchar(60) NOT NULL\n);\n\nCREATE TABLE departments (\n    department varchar(60) PRIMARY KEY,\n    service_line varchar(80) NOT NULL\n);\n\nCREATE TABLE admissions (\n    admission_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    admit_date date NOT NULL,\n    discharge_date date,\n    diagnosis varchar(120) NOT NULL,\n    department varchar(60) NOT NULL REFERENCES departments(department),\n    CHECK (discharge_date IS NULL OR discharge_date >= admit_date)\n);\n\nCREATE TABLE lab_results (\n    test_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    test_name varchar(60) NOT NULL,\n    result_value numeric(10,2),\n    test_date date NOT NULL,\n    normal_low numeric(10,2),\n    normal_high numeric(10,2)\n);\n\nCREATE TABLE prescriptions (\n    rx_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    medication varchar(100) NOT NULL,\n    dosage varchar(50) NOT NULL,\n    prescribed_date date NOT NULL,\n    duration_days integer CHECK (duration_days > 0)\n);\n\n-- The NULL is deliberate: it lets students see why NOT IN can be unsafe.\nCREATE TABLE review_queue (\n    queue_id integer PRIMARY KEY,\n    patient_id integer\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith',45,'Female','Greensboro'),\n(1002,'Bob Johnson',32,'Male','High Point'),\n(1003,'Carol White',58,'Female','Greensboro'),\n(1004,'David Brown',67,'Male','Burlington'),\n(1005,'Eve Davis',29,'Female','Greensboro'),\n(1006,'Frank Miller',71,'Male','Reidsville'),\n(1007,'Grace Wilson',52,'Female','Greensboro'),\n(1008,'Henry Taylor',48,'Male','High Point'),\n(1009,'Ivy Clark',63,'Female','Greensboro'),\n(1010,'Jack Anderson',55,'Male','Burlington'),\n(1011,'Kara Young',41,'Female','Greensboro');\n\nINSERT INTO departments VALUES\n('Cardiology','Cardiovascular'),\n('Endocrinology','Medicine'),\n('Internal Medicine','Medicine'),\n('Neurology','Neurosciences'),\n('Orthopedics','Musculoskeletal'),\n('Pulmonology','Medicine');\n\nINSERT INTO admissions VALUES\n(101,1001,'2026-01-15','2026-01-20','Pneumonia','Internal Medicine'),\n(102,1002,'2026-01-16','2026-01-18','Fractured arm','Orthopedics'),\n(103,1003,'2026-01-17','2026-01-25','Cardiac arrhythmia','Cardiology'),\n(104,1001,'2026-03-05','2026-03-12','Hypertension','Internal Medicine'),\n(105,1004,'2026-02-19','2026-02-26','Heart disease','Cardiology'),\n(106,1005,'2026-04-20','2026-04-22','Migraine','Neurology'),\n(107,1006,'2026-05-21','2026-05-28','Diabetes complications','Endocrinology'),\n(108,1007,'2026-05-22','2026-05-24','Asthma','Pulmonology'),\n(109,1008,'2026-06-23','2026-06-30','Cardiac monitoring','Cardiology'),\n(110,1009,'2026-07-24','2026-07-27','Joint pain','Orthopedics'),\n(111,1010,'2026-07-25',NULL,'Hypertension','Internal Medicine'),\n(112,1006,'2026-08-01','2026-08-05','Cardiac assessment','Cardiology');\n\nINSERT INTO lab_results VALUES\n(201,1001,'Glucose',98,'2026-01-16',70,110),\n(202,1002,'Cholesterol',185,'2026-01-17',0,200),\n(203,1003,'Glucose',145,'2026-01-18',70,110),\n(204,1004,'Systolic BP',150,'2026-02-19',90,140),\n(205,1005,'Cholesterol',210,'2026-04-20',0,200),\n(206,1006,'Glucose',320,'2026-05-21',70,110),\n(207,1007,'Systolic BP',128,'2026-05-22',90,140),\n(208,1008,'Cholesterol',195,'2026-06-23',0,200),\n(209,1009,'Glucose',110,'2026-07-24',70,110),\n(210,1010,'Systolic BP',142,'2026-07-25',90,140),\n(211,1001,'Hemoglobin',14.2,'2026-01-16',12,16),\n(212,1003,'Hemoglobin',11.8,'2026-01-18',12,16),\n(213,1006,'Hemoglobin',10.9,'2026-05-21',12,16);\n\nINSERT INTO prescriptions VALUES\n(301,1001,'Amoxicillin','500mg','2026-01-16',7),\n(302,1002,'Ibuprofen','400mg','2026-01-17',14),\n(303,1003,'Metoprolol','25mg','2026-01-18',90),\n(304,1004,'Atorvastatin','40mg','2026-02-19',30),\n(305,1006,'Metformin','1000mg','2026-05-21',60),\n(306,1006,'Insulin','50 units','2026-05-21',90),\n(307,1001,'Lisinopril','5mg','2026-03-06',30),\n(308,1007,'Albuterol','2 puffs','2026-05-22',15),\n(309,1008,'Aspirin','81mg','2026-06-23',365),\n(310,1009,'Ibuprofen','400mg','2026-07-24',7);\n\nINSERT INTO review_queue VALUES\n(1,1003),\n(2,NULL);"

execute_script(setup_sql)


# MODULE EXAMPLES


## Preview patients

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, name, age, home_city\nFROM patients\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Preview patients")


## Patients older than average

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, name, age\nFROM patients\nWHERE age > (SELECT AVG(age) FROM patients)\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Patients older than average")


## Show the overall average beside each patient

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, name, age,\n       (SELECT ROUND(AVG(age),1) FROM patients) AS hospital_avg_age\nFROM patients\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Show the overall average beside each patient")


## No matching inner row

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT (\n  SELECT result_value\n  FROM lab_results\n  WHERE test_id = 9999\n) AS missing_scalar_result;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="No matching inner row")


## Invalid scalar result

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- A scalar subquery must produce at most one row.\nSELECT (SELECT patient_id FROM admissions) AS patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Invalid scalar result")


## Patient IDs in the Cardiology set

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, name, age\nFROM patients\nWHERE patient_id IN (\n  SELECT patient_id\n  FROM admissions\n  WHERE department = 'Cardiology'\n)\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Patient IDs in the Cardiology set")


## Compare IN and JOIN

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- IN answers membership and returns each patient once.\nSELECT p.patient_id, p.name\nFROM patients AS p\nWHERE p.patient_id IN (SELECT a.patient_id FROM admissions AS a)\nORDER BY p.patient_id;\n\n-- A JOIN returns matching row combinations; repeat admissions can repeat a patient.\nSELECT p.patient_id, p.name, a.admission_id\nFROM patients AS p\nJOIN admissions AS a ON a.patient_id = p.patient_id\nORDER BY p.patient_id, a.admission_id;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## NOT IN against review_queue

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- review_queue contains patient 1003 AND a NULL patient_id.\n-- Because no comparison is definitively TRUE for many rows, NOT IN becomes UNKNOWN.\nSELECT patient_id, name\nFROM patients\nWHERE patient_id NOT IN (SELECT patient_id FROM review_queue)\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="NOT IN against review_queue")


## Use NOT EXISTS

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name\nFROM patients AS p\nWHERE NOT EXISTS (\n  SELECT 1\n  FROM review_queue AS q\n  WHERE q.patient_id = p.patient_id\n)\nORDER BY p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Use NOT EXISTS")


## Abnormal lab exists

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, p.age\nFROM patients AS p\nWHERE EXISTS (\n  SELECT 1\n  FROM lab_results AS l\n  WHERE l.patient_id = p.patient_id\n    AND (l.result_value < l.normal_low OR l.result_value > l.normal_high)\n)\nORDER BY p.age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Abnormal lab exists")


## No prescriptions exist

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name\nFROM patients AS p\nWHERE NOT EXISTS (\n  SELECT 1 FROM prescriptions AS pr\n  WHERE pr.patient_id = p.patient_id\n)\nORDER BY p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="No prescriptions exist")


## age > ANY (...)

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, p.age\nFROM patients AS p\nWHERE p.age > ANY (\n  SELECT p2.age\n  FROM patients AS p2\n  JOIN admissions AS a ON a.patient_id = p2.patient_id\n  WHERE a.department = 'Cardiology'\n)\nORDER BY p.age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="age > ANY (...)")


## age > ALL (...)

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, p.age\nFROM patients AS p\nWHERE p.age > ALL (\n  SELECT p2.age\n  FROM patients AS p2\n  JOIN admissions AS a ON a.patient_id = p2.patient_id\n  WHERE a.department = 'Cardiology'\n)\nORDER BY p.age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="age > ALL (...)")


## Compare ANY and ALL with an empty set

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT\n  45 > ANY (SELECT age FROM patients WHERE patient_id < 0) AS greater_than_any_empty,\n  45 > ALL (SELECT age FROM patients WHERE patient_id < 0) AS greater_than_all_empty;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Compare ANY and ALL with an empty set")


## Above the average for the same test

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT l.test_id, l.patient_id, l.test_name, l.result_value,\n       ROUND((\n         SELECT AVG(l2.result_value)\n         FROM lab_results AS l2\n         WHERE l2.test_name = l.test_name\n       ),2) AS test_average\nFROM lab_results AS l\nWHERE l.result_value > (\n  SELECT AVG(l3.result_value)\n  FROM lab_results AS l3\n  WHERE l3.test_name = l.test_name\n)\nORDER BY l.test_name, l.result_value DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Above the average for the same test")


## Admission count per patient

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name,\n       (SELECT COUNT(*)\n        FROM admissions AS a\n        WHERE a.patient_id = p.patient_id) AS admission_count\nFROM patients AS p\nORDER BY admission_count DESC, p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Admission count per patient")


## Latest admission per patient

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, a.admission_id, a.admit_date, a.diagnosis\nFROM patients AS p\nJOIN admissions AS a ON a.patient_id = p.patient_id\nWHERE a.admit_date = (\n  SELECT MAX(a2.admit_date)\n  FROM admissions AS a2\n  WHERE a2.patient_id = p.patient_id\n)\nORDER BY p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Latest admission per patient")


## Admission counts as a derived table

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, COALESCE(ac.admission_count,0) AS admission_count\nFROM patients AS p\nLEFT JOIN (\n  SELECT patient_id, COUNT(*) AS admission_count\n  FROM admissions\n  GROUP BY patient_id\n) AS ac ON ac.patient_id = p.patient_id\nORDER BY admission_count DESC, p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Admission counts as a derived table")


## Pre-aggregate then join

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- Same goal as a correlated count, expressed with pre-aggregation + join.\nSELECT p.patient_id, p.name, COALESCE(a.admission_count,0) AS admission_count\nFROM patients AS p\nLEFT JOIN (\n  SELECT patient_id, COUNT(*) AS admission_count\n  FROM admissions\n  GROUP BY patient_id\n) AS a USING (patient_id)\nORDER BY admission_count DESC, p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Pre-aggregate then join")


## EXPLAIN an EXISTS query

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "EXPLAIN\nSELECT p.patient_id, p.name\nFROM patients AS p\nWHERE EXISTS (\n  SELECT 1 FROM admissions AS a\n  WHERE a.patient_id = p.patient_id\n);\n\n\n-- ------------------------------------------------------------"

display_query(query, title="EXPLAIN an EXISTS query")


## Average-age condition

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, name, age\nFROM patients\nWHERE age > (SELECT AVG(age) FROM patients)\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Average-age condition")


## Admission count per patient

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name,\n       (SELECT COUNT(*)\n        FROM admissions AS a\n        WHERE a.patient_id = p.patient_id) AS admission_count\nFROM patients AS p\nORDER BY admission_count DESC, p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Admission count per patient")


## Abnormal lab exists

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, p.age\nFROM patients AS p\nWHERE EXISTS (\n  SELECT 1\n  FROM lab_results AS l\n  WHERE l.patient_id = p.patient_id\n    AND (l.result_value < l.normal_low OR l.result_value > l.normal_high)\n)\nORDER BY p.age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Abnormal lab exists")


## Distinct medication count

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name,\n       (SELECT COUNT(DISTINCT pr.medication)\n        FROM prescriptions AS pr\n        WHERE pr.patient_id = p.patient_id) AS medication_count\nFROM patients AS p\nORDER BY medication_count DESC, p.patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Distinct medication count")


## Complex-patient query

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, p.age\nFROM patients AS p\nWHERE p.age > (SELECT AVG(age) FROM patients)\n  AND (SELECT COUNT(*) FROM admissions AS a WHERE a.patient_id = p.patient_id) > 1\n  AND EXISTS (\n      SELECT 1 FROM lab_results AS l\n      WHERE l.patient_id = p.patient_id\n        AND (l.result_value < l.normal_low OR l.result_value > l.normal_high)\n  )\n  AND (SELECT COUNT(DISTINCT pr.medication)\n       FROM prescriptions AS pr\n       WHERE pr.patient_id = p.patient_id) > 1\nORDER BY p.age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Complex-patient query")


## Patient feature table

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH patient_features AS (\n  SELECT p.patient_id, p.name, p.age,\n         (SELECT COUNT(*) FROM admissions a WHERE a.patient_id=p.patient_id) AS admissions,\n         (SELECT COUNT(*) FROM lab_results l WHERE l.patient_id=p.patient_id AND (l.result_value<l.normal_low OR l.result_value>l.normal_high)) AS abnormal_labs,\n         (SELECT COUNT(DISTINCT medication) FROM prescriptions pr WHERE pr.patient_id=p.patient_id) AS medications\n  FROM patients p\n)\nSELECT *,\n       (CASE WHEN age >= 65 THEN 3 WHEN age >= 50 THEN 2 ELSE 1 END\n        + CASE WHEN admissions >= 2 THEN 2 WHEN admissions = 1 THEN 1 ELSE 0 END\n        + LEAST(abnormal_labs,2)\n        + LEAST(medications,2)) AS risk_score\nFROM patient_features\nORDER BY risk_score DESC, patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Patient feature table")


## Department dashboard

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT a.department,\n       COUNT(*) AS admissions,\n       COUNT(DISTINCT a.patient_id) AS patients,\n       ROUND(AVG(a.discharge_date - a.admit_date),1) AS avg_stay_days,\n       (SELECT COUNT(*)\n        FROM admissions AS x\n        WHERE x.department = a.department\n          AND x.discharge_date IS NULL) AS current_open_admissions\nFROM admissions AS a\nGROUP BY a.department\nORDER BY admissions DESC, a.department;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Department dashboard")


## Assessment starter

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- Assessment: return patients whose admission count is above\n-- the average admission count among patients who have at least one admission.\n-- Use a derived table and at least one subquery.\n\nSELECT p.patient_id, p.name\nFROM patients AS p\n-- complete the query\nORDER BY p.patient_id;"

display_query(query, title="Assessment starter")


# SUBQUERY DECISION GUIDE


Need one value?          -> scalar subquery
Need a list of values?   -> IN or a multi-row subquery
Need existence only?     -> EXISTS / NOT EXISTS
Need row-specific logic? -> correlated subquery
Need a named stage?      -> CTE
Need a temporary relation in FROM? -> derived table


# VALIDATION HABITS


[ ] Know whether the subquery should return one value or many.
[ ] Test the inner query by itself first.
[ ] Watch for NULL behavior with NOT IN.
[ ] Prefer EXISTS / NOT EXISTS for existence logic.
[ ] Confirm the outer-query grain after correlation.


# Module 7 summary

Before moving on, make sure you can explain the main patterns in your own words.

For any query, ask:
1. What question does it answer?
2. What does one input row represent?
3. What does one output row represent?
4. What SQL rule changed or filtered the data?
5. How would you validate the result?

## Next
Continue to **Module 8**.
